# Dataset Pre-Processing & Splitting
This notebook reads the raw global dataset from `all_competitions_events.pkl`, applies the xT preprocessing steps, and splits the data into train and test sets (70/30 by match_id). The resulting datasets are saved back to disk to serve as the baseline for all subsequent model training and manual evaluation processes.

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Setup paths to import from src
framework_dir = os.path.dirname(os.getcwd())
if framework_dir not in sys.path:
    sys.path.append(framework_dir)
    sys.path.append(os.path.join(framework_dir, 'src'))

from src import config
from src.engine.xt_model import prepare_xt_data

## 1. Load Raw Global Data
Loads the monolithic events pickle.

In [2]:
print(f"Loading raw dataset from {config.GLOBAL_DATA_FILE}...")
if os.path.exists(config.GLOBAL_DATA_FILE):
    df_raw = pd.read_pickle(config.GLOBAL_DATA_FILE)
    print(f"Loaded {len(df_raw)} raw events.")
else:
    print("Error: Raw dataset not found. Please ensure it has been downloaded first.")

Loading raw dataset from D:\MSC DS ABDN\PROJ_The science of football - Jan 2026\CODE\Vibe\football_analytics_framework\data\all_competitions_events.pkl...
Loaded 6108353 raw events.


## 2. Pre-process Data
Standardize coordinates, filter for passes/carries, and inject missing values needed for xT matrix and TransGoalNet graphs.

In [3]:
print("Applying xT data pre-processing (prepare_xt_data)...")
actions_df = prepare_xt_data(df_raw)
print(f"Filtered and prepared {len(actions_df)} action events.")

# Clear raw dataset from memory
del df_raw
import gc
gc.collect()

Applying xT data pre-processing (prepare_xt_data)...
Filtered and prepared 6108353 action events.


0

## 3. Extract Train/Test Subset
We extract 500 matches for training and 100 matches for testing.


In [5]:
print("Selecting 500 matches for Train, 100 for Test...")
unique_matches = actions_df['match_id'].unique()
np.random.seed(42)
np.random.shuffle(unique_matches)

train_matches = unique_matches[:500]
test_matches = unique_matches[500:600]

train_actions = actions_df[actions_df['match_id'].isin(train_matches)].copy()
test_actions = actions_df[actions_df['match_id'].isin(test_matches)].copy()

print(f"Train matches: {len(train_matches)} | Train events: {len(train_actions)}")
print(f"Test matches:  {len(test_matches)} | Test events:  {len(test_actions)}")

Selecting 500 matches for Train, 100 for Test...
Train matches: 500 | Train events: 887342
Test matches:  100 | Test events:  176551


## 4. Save to Disk
Export to the pre-configured paths so they can be read instantaneously by `evaluate_all_models.py` and `Manual_Model_Evaluation.ipynb`.

In [6]:
print(f"Saving Train Split to: {config.TRAIN_ACTIONS_FILE}")
train_actions.to_pickle(config.TRAIN_ACTIONS_FILE)

print(f"Saving Test Split to:  {config.TEST_ACTIONS_FILE}")
test_actions.to_pickle(config.TEST_ACTIONS_FILE)

print("✅ Splits successfully saved to disk.")

Saving Train Split to: D:\MSC DS ABDN\PROJ_The science of football - Jan 2026\CODE\Vibe\football_analytics_framework\data\train_actions.pkl
Saving Test Split to:  D:\MSC DS ABDN\PROJ_The science of football - Jan 2026\CODE\Vibe\football_analytics_framework\data\test_actions.pkl
✅ Splits successfully saved to disk.
